In [1]:
pip install pymupdf

Note: you may need to restart the kernel to use updated packages.


In [2]:
# ============================================================
# Automotive Feature Spec → Test Case CSV Generator
# Generates test cases in hierarchical step format matching sample
#
# Install dependencies:
#   pip install openai pandas openpyxl pdfplumber
#
# Files needed in the same folder as this notebook:
#   - config.py       (contains: api_key = "sk-...")
#   - your_spec.pdf
#   - smaple.xlsx
# ============================================================

import openai
import pandas as pd
import pdfplumber
import csv
import json
import os
import re
import config

# ─────────────────────────────────────────
# CONFIGURATION
# ─────────────────────────────────────────

openai.api_key = config.api_key

SPEC_PDF_PATH    = "Automotive_Feature_Spec_ACC_260502_113241.pdf"
SAMPLE_XLSX_PATH = "smaple.xlsx"
OUTPUT_CSV_PATH  = "test_cases_generated.csv"
MODEL            = "gpt-4o"

# ─────────────────────────────────────────
# STEP 1 — Extract text from PDF
# ─────────────────────────────────────────

def extract_text_from_pdf(pdf_path):
    pages = []
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            text = page.extract_text()
            if text:
                pages.append(text)
    return "\n\n".join(pages)

# ─────────────────────────────────────────
# STEP 2 — Read columns from xlsx
# ─────────────────────────────────────────

def get_columns(sample_path):
    df = pd.read_excel(sample_path, sheet_name=0)
    return list(df.columns)

# ─────────────────────────────────────────
# STEP 3 — Prompt
# ─────────────────────────────────────────

SYSTEM_PROMPT = """You are a senior automotive test engineer who has been writing test cases
for ADAS systems for over 10 years. You write test cases the way an experienced engineer
would — not like a template or a machine.

WRITING STYLE RULES (very important):
- Write steps the way a real tester would write them — direct, practical, to the point
- Avoid robotic/templated phrases like "System detects the object and begins verification"
  Instead write: "Check radar output log — RCS value should cross threshold"
- Avoid starting every sentence with "System..." or "The system..."
- Use real automotive signal names where relevant (e.g. BCM, TTC, RCS, FoV, Yaw-rate, CAN-FD)
- Mix short and longer sentences naturally — not everything the same length
- Preconditions should read like a tester setting up the bench: "Set ego speed to 100 km/h, ACC engaged, lane clear"
- Steps should read like instructions to another tester: "Inject cut-in vehicle at 80 km/h, 20m gap"
- Expected outputs should be specific and measurable: "Deceleration request of -3.0 m/s² sent to BCM within 200ms"
- Postconditions should describe the final stable state: "Ego speed stabilized, safe following distance maintained, no DTC set"
- Do NOT repeat the same sentence structure in every row
- Vary the language — sometimes say "Verify", sometimes "Confirm", sometimes "Check", sometimes "Ensure"

FORMAT RULES:

Each test case = multiple CSV rows structured as:

Row 1 — Test Case Header:
  XP-nnn Source id  = XP-100_001
  Test Case Detail Type = XP-100_001
  Step Description  = <test case name>
  Input conditions  = <one line summary of what this test validates>
  Expected Output, Input signals, Output Signals = empty

Row 2 — Precondition:
  XP-nnn Source id  = XP-100_001_01
  Test Case Detail Type = XP-100_001_01
  Step Description  = Precondition
  Input conditions  = <bench/vehicle setup — written like a real tester would set it up>
  Expected Output   = <what needs to be true before test starts>
  Input signals, Output Signals = empty

Rows 3..N — Steps (2 to 4 steps per test case):
  XP-nnn Source id  = XP-100_001_02, _03, _04 ...
  Test Case Detail Type = same
  Step Description  = <short action verb phrase — e.g. "Inject cut-in vehicle", "Monitor BCM request">
  Input conditions  = <what the tester does or injects>
  Expected Output   = <specific measurable expected result>
  Input signals     = <signal names being stimulated, if any>
  Output Signals    = <signal names being observed, if any>

Last Row — Postcondition:
  XP-nnn Source id  = XP-100_001_last
  Test Case Detail Type = XP-100_001_last
  Step Description  = Postcondition
  Input conditions  = <final system state after test>
  Expected Output   = <confirmation of pass criteria>
  Input signals, Output Signals = empty

NUMBERING:
- Test cases: XP-100_001, XP-100_002, XP-100_003 ...
- Steps: XP-100_001_01, XP-100_001_02 ... XP-100_001_last

COVERAGE — generate one test case for each:
- Every Functional Requirement (REQ_ACC_001, REQ_ACC_002, etc.)
- Every Operational Scenario (Scenario 1, 2, 3, 4 ...)
- Every Interface/Diagnostic item (CAN-FD, DTC P1500, DID 0x4001, DID 0x4002)

Return ONLY valid JSON: { "rows": [ {...}, {...} ] }
Each object has exactly these keys:
"XP-nnn Source id", "Test Case Detail Type", "Step Description", "Input conditions", "Expected Output", "Input signals", "Output Signals"
"""

def build_prompt(spec_text, columns):
    return f"""
Here is the automotive feature specification. Generate test cases for every item in it.

Write everything the way an experienced automotive test engineer would — not like a template.
Steps should be direct and practical. Use real signal/module names. Vary sentence structure.

FEATURE SPECIFICATION:
{spec_text}

Return ONLY this JSON — no markdown, no explanation:
{{
  "rows": [
    {{"XP-nnn Source id": "XP-100_001", "Test Case Detail Type": "XP-100_001", "Step Description": "High-Speed Cut-In", "Input conditions": "Validates ACC deceleration response when a vehicle cuts in at 80 km/h from adjacent lane", "Expected Output": "", "Input signals": "", "Output Signals": ""}},
    ...continue for all rows of all test cases...
  ]
}}
"""

# ─────────────────────────────────────────
# STEP 4 — Call OpenAI
# ─────────────────────────────────────────

def generate_rows(spec_text, columns):
    print(f"[INFO] Calling OpenAI {MODEL} ...")

    response = openai.ChatCompletion.create(
        model=MODEL,
        max_tokens=8000,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": build_prompt(spec_text, columns)},
        ],
    )

    raw = response["choices"][0]["message"]["content"].strip()

    # Strip markdown fences if model adds them
    if raw.startswith("```"):
        raw = raw.strip("```").strip()
        if raw.startswith("json"):
            raw = raw[4:].strip()

    parsed = json.loads(raw)

    # Unwrap wrapper
    if isinstance(parsed, dict):
        for v in parsed.values():
            if isinstance(v, list):
                parsed = v
                break

    if not isinstance(parsed, list):
        raise ValueError(f"Unexpected response format: {type(parsed)}")

    print(f"[INFO] Generated {len(parsed)} rows.")
    return parsed

# ─────────────────────────────────────────
# STEP 5 — Save to CSV
# ─────────────────────────────────────────

def save_to_csv(rows, output_path):
    fieldnames = [
        "XP-nnn Source id",
        "Test Case Detail Type",
        "Step Description",
        "Input conditions",
        "Expected Output",
        "Input signals",
        "Output Signals"
    ]

    with open(output_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames, extrasaction="ignore")

        # Write the two header rows exactly like the sample
        writer.writerow({
            "XP-nnn Source id":      "XP-nnn Source id",
            "Test Case Detail Type": "Test Case Detail Type",
            "Step Description":      "Step Description",
            "Input conditions":      "Input conditions",
            "Expected Output":       "Expected Output",
            "Input signals":         "Input signals",
            "Output Signals":        "Output Signals"
        })
        writer.writerow({
            "XP-nnn Source id":      "Test case ID",
            "Test Case Detail Type": "Test Case Shell",
            "Step Description":      "Test Name",
            "Input conditions":      "Test Description",
            "Expected Output":       "Dspace",
            "Input signals":         "VMM",
            "Output Signals":        "etc"
        })

        # Write all generated rows — insert empty separator row between test cases
        empty_row = {col: "" for col in fieldnames}
        for i, row in enumerate(rows):
            src_id = row.get("XP-nnn Source id", "")
            # Detect start of a new test case (ID like XP-100_001 with no step suffix)
            # Insert empty row before it, but not before the very first test case
            if re.match(r'^XP-\d+_\d{3}$', src_id) and i != 0:
                writer.writerow(empty_row)
            writer.writerow({col: row.get(col, "") for col in fieldnames})

    print(f"[INFO] CSV saved → {output_path}")

# ─────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────

print("[STEP 1] Reading feature spec PDF ...")
spec_text = extract_text_from_pdf(SPEC_PDF_PATH)


print("[STEP 2] Reading sample columns from xlsx ...")
columns = get_columns(SAMPLE_XLSX_PATH)


print("[STEP 3] Generating test cases via OpenAI ...")
rows = generate_rows(spec_text, columns)

print("[STEP 4] Preview — first 10 rows:")

print("[STEP 5] Saving to CSV ...")
save_to_csv(rows, OUTPUT_CSV_PATH)
print("\n[DONE] All steps completed successfully.")


[STEP 1] Reading feature spec PDF ...
[STEP 2] Reading sample columns from xlsx ...
[STEP 3] Generating test cases via OpenAI ...
[INFO] Calling OpenAI gpt-4o ...
[INFO] Generated 31 rows.
[STEP 4] Preview — first 10 rows:
[STEP 5] Saving to CSV ...
[INFO] CSV saved → test_cases_generated.csv

[DONE] All steps completed successfully.
